# 🔍 Deepfake Detector — Free Colab Optimized
**3-Class:** Real | AI Generated | Deepfake

> ⚠️ Before running: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 — Check GPU
# ═══════════════════════════════════════════════
import torch, sys

print('='*55)
print('  DEEPFAKE DETECTOR — FREE COLAB OPTIMIZED')
print('='*55)

if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU')

gpu = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'✅ GPU    : {gpu.name}')
print(f'✅ VRAM   : {vram:.1f} GB')
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ Python : {sys.version.split()[0]}')
print('='*55)

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 — Mount Google Drive
# ═══════════════════════════════════════════════
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

MODEL_SAVE = Path('/content/drive/MyDrive/AI-Deepfake-Detector/models')
MODEL_SAVE.mkdir(parents=True, exist_ok=True)
CKPT_PATH = MODEL_SAVE / 'best_model.pth'

print(f'✅ Model save dir: {MODEL_SAVE}')
if CKPT_PATH.exists():
    mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f'📁 Checkpoint found ({mb:.0f} MB) → will auto-resume!')
else:
    print('📁 No checkpoint found → training from scratch')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3 — Extract Dataset (auto-detects structure)
# Archive : /MyDrive/AI-Deepfake-Detector/unified_dataset.7z
# ═══════════════════════════════════════════════
import os, zipfile
from pathlib import Path

ARCHIVE_PATH = '/content/drive/MyDrive/AI-Deepfake-Detector/unified_dataset.7z'
EXTRACT_DIR  = Path('/content/dataset_raw')
EXTENSIONS   = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# ── Step 1: Extract ──────────────────────────────
if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.rglob('*')):
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    if not Path(ARCHIVE_PATH).exists():
        raise FileNotFoundError(
            f'❌ Archive not found at:\n   {ARCHIVE_PATH}\n'
            'Please check the path in Google Drive.'
        )
    size_gb = os.path.getsize(ARCHIVE_PATH) / 1e9
    print(f'📦 Archive found: {size_gb:.2f} GB')
    print('📋 Copying to local SSD (faster extraction)...')
    local_archive = '/content/archive.7z'
    os.system(f'cp "{ARCHIVE_PATH}" {local_archive}')
    print('📦 Installing p7zip and extracting...')
    os.system('apt-get install -y p7zip-full -qq')
    ret = os.system(f'7z x {local_archive} -o{EXTRACT_DIR} -y -bd')
    os.remove(local_archive)
    if ret != 0:
        raise RuntimeError('❌ Extraction failed! Check your 7z file.')
    print('✅ Extraction complete!')
else:
    print('✅ Already extracted — skipping.')

# ── Step 2: Auto-detect folder structure ────────
FOLDER_NAMES = ['real', 'ai_generated', 'deepfake']

def find_class_dirs(root: Path):
    """Walk the extracted tree and find real/ai_generated/deepfake folders."""
    found = {}
    for folder in FOLDER_NAMES:
        # Search up to 3 levels deep
        matches = list(root.rglob(folder))
        # Keep only directories that have images
        img_dirs = [m for m in matches if m.is_dir() and
                    any(f.suffix.lower() in EXTENSIONS for f in m.rglob('*'))]
        if img_dirs:
            # Prefer shallowest match
            found[folder] = sorted(img_dirs, key=lambda p: len(p.parts))[0]
    return found

print('\n🔍 Auto-detecting dataset folder structure...')
CLASS_DIRS = find_class_dirs(EXTRACT_DIR)

if len(CLASS_DIRS) == 0:
    # List what we actually got
    print('⚠️  Could not find expected folders. Contents of extracted dir:')
    for p in sorted(EXTRACT_DIR.rglob('*'))[:30]:
        print(f'   {p.relative_to(EXTRACT_DIR)}')
    raise RuntimeError(
        'Expected folders named: real, ai_generated, deepfake\n'
        'Please rename your dataset folders to match.'
    )

print('\n📊 Dataset Statistics:')
TOTAL_IMAGES = {}
for cls, d in CLASS_DIRS.items():
    imgs = [f for f in d.rglob('*') if f.suffix.lower() in EXTENSIONS]
    TOTAL_IMAGES[cls] = len(imgs)
    print(f'  {cls:>14}: {len(imgs):>7,} images  →  {d.relative_to(EXTRACT_DIR)}')

missing = [c for c in FOLDER_NAMES if c not in CLASS_DIRS]
if missing:
    print(f'\n⚠️  Missing folders: {missing}')
    print('   Training will use only the found classes.')
else:
    print('\n✅ All 3 classes found — ready to train!')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4 — Install & Import
# ═══════════════════════════════════════════════
!pip install -q scikit-learn matplotlib seaborn tqdm

import torch, torch.nn as nn, torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from PIL import Image
import numpy as np, copy, time, os, random
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda')
print('✅ All imports OK')
print(f'✅ Device: {device} ({torch.cuda.get_device_name(0)})')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5 — Config & Dataset Class
#
# FREE COLAB SAFE SETTINGS:
#   EfficientNet-B0  → fits T4 comfortably (B4 risks OOM)
#   BATCH_SIZE = 16  → safe; raise to 32 only if no OOM
#   MAX_PER_CLASS = 5000 → keeps RAM < 8 GB
#   NUM_WORKERS = 0  → avoids Colab multiprocessing crash
# ═══════════════════════════════════════════════
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

# Build label map from what was actually found in Cell 3
LABEL_MAP    = {folder: idx for idx, folder in enumerate(sorted(CLASS_DIRS.keys()))}
CLASS_NAMES  = [k for k in sorted(CLASS_DIRS.keys())]
NUM_CLASSES  = len(CLASS_NAMES)
EXTENSIONS   = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

IMG_SIZE      = 224
RESIZE_TO     = 232
MEAN          = [0.485, 0.456, 0.406]
STD           = [0.229, 0.224, 0.225]
BATCH_SIZE    = 16    # Safe for free T4 — increase to 32 if VRAM allows
NUM_WORKERS   = 0     # Must be 0 on free Colab
MAX_PER_CLASS = 5000  # Raise to 10000 if RAM allows
VAL_SPLIT     = 0.15

print(f'Classes found   : {CLASS_NAMES}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Max/class       : {MAX_PER_CLASS:,}')
print(f'Workers         : {NUM_WORKERS}')

class DeepfakeDataset(Dataset):
    def __init__(self, class_dirs, split='train', seed=42):
        self.transform = self._make_transforms(split)
        self.samples   = []
        rng = np.random.default_rng(seed)
        print(f'\n📂 Building [{split}] split...')
        for folder, label in LABEL_MAP.items():
            d    = class_dirs[folder]
            imgs = sorted([p for p in d.rglob('*') if p.suffix.lower() in EXTENSIONS])
            rng.shuffle(imgs)
            imgs   = imgs[:MAX_PER_CLASS]
            n_val  = max(1, int(len(imgs) * VAL_SPLIT))
            chosen = imgs[:n_val] if split == 'val' else imgs[n_val:]
            self.samples += [(str(p), label) for p in chosen]
            print(f'  {CLASS_NAMES[label]:>14}: {len(chosen):>5,}')
        if not self.samples:
            raise RuntimeError('No images loaded! Check CLASS_DIRS.')
        print(f'  ✅ Total {split}: {len(self.samples):,}')

    @staticmethod
    def _make_transforms(split):
        if split == 'train':
            return transforms.Compose([
                transforms.Resize(RESIZE_TO),
                transforms.RandomCrop(IMG_SIZE),
                transforms.RandomHorizontalFlip(0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])
        return transforms.Compose([
            transforms.Resize(RESIZE_TO),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), label
        except Exception:
            return self.transform(Image.new('RGB', (IMG_SIZE, IMG_SIZE))), label

    def get_class_weights(self):
        labels = [s[1] for s in self.samples]
        counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
        counts = np.where(counts == 0, 1, counts)
        w = 1.0 / counts
        return torch.FloatTensor([w[l] for l in labels])

train_ds = DeepfakeDataset(CLASS_DIRS, split='train')
val_ds   = DeepfakeDataset(CLASS_DIRS, split='val')

sampler = WeightedRandomSampler(
    weights=train_ds.get_class_weights(),
    num_samples=len(train_ds), replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'\n✅ Train batches: {len(train_loader):,}')
print(f'✅ Val   batches: {len(val_loader):,}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 — Model: EfficientNet-B0
# Why B0 not B4?
#   B4 @ BS=16 → ~7-8 GB VRAM → risky OOM on free T4
#   B0 @ BS=16 → ~2 GB VRAM   → comfortable, faster epochs
#   Accuracy gap is small (~1-2%) — not worth the crash risk
# ═══════════════════════════════════════════════
import torch.nn as nn
from torchvision import models

class DeepfakeDetector(nn.Module):
    def __init__(self, num_classes=3, dropout=0.4):
        super().__init__()
        self.backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )
        in_f = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(in_f, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(p=dropout / 2, inplace=True),
            nn.Linear(256, num_classes),
        )
        self.class_names = CLASS_NAMES
        self.num_classes  = num_classes

    def forward(self, x):
        return self.backbone(x)

    def freeze_backbone(self):
        for p in self.backbone.features.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.features.parameters(): p.requires_grad = True


model = DeepfakeDetector(NUM_CLASSES).to(device)
model.freeze_backbone()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Architecture  : EfficientNet-B0')
print(f'✅ Total params  : {total:,}')
print(f'✅ Trainable     : {trainable:,}  (backbone frozen for warm-up)')

torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1e9
total_v = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ VRAM used     : {used:.2f} / {total_v:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 7 — Train & Validate Functions
# ═══════════════════════════════════════════════
from tqdm.notebook import tqdm
from torch.amp import autocast

def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = correct = total = 0
    cc = [0]*NUM_CLASSES; ct = [0]*NUM_CLASSES
    pbar = tqdm(loader, desc='Train', leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        pred    = out.argmax(1)
        total  += labels.size(0)
        correct += pred.eq(labels).sum().item()
        for c in range(NUM_CLASSES):
            m = labels == c
            if m.any():
                cc[c] += pred[m].eq(labels[m]).sum().item()
                ct[c] += m.sum().item()
        pbar.set_postfix(loss=f'{loss.item():.3f}', acc=f'{100*correct/total:.1f}%')
    per_cls = [cc[c] / max(ct[c], 1) for c in range(NUM_CLASSES)]
    return total_loss / total, correct / total, per_cls


def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    cc = [0]*NUM_CLASSES; ct = [0]*NUM_CLASSES
    all_preds, all_labels = [], []
    with torch.no_grad():
        pbar = tqdm(loader, desc='Val  ', leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            with autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            pred    = out.argmax(1)
            total  += labels.size(0)
            correct += pred.eq(labels).sum().item()
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            for c in range(NUM_CLASSES):
                m = labels == c
                if m.any():
                    cc[c] += pred[m].eq(labels[m]).sum().item()
                    ct[c] += m.sum().item()
            pbar.set_postfix(loss=f'{loss.item():.3f}', acc=f'{100*correct/total:.1f}%')
    per_cls = [cc[c] / max(ct[c], 1) for c in range(NUM_CLASSES)]
    return total_loss / total, correct / total, per_cls, all_preds, all_labels


print('✅ Train / validate functions ready')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 8 — Training Loop (with auto-resume)
#
# FREE COLAB TIPS:
#  • Sessions timeout after ~90 min idle / ~4-5 hr total
#  • Checkpoint auto-saves to Drive after every improvement
#  • If disconnected: re-run cells 1→7 then re-run this cell
#    — it will resume from last saved checkpoint automatically
#  • To prevent idle timeout, open browser Console (F12) and run:
#    setInterval(() => document.querySelector('colab-connect-button').click(), 60000)
# ═══════════════════════════════════════════════
import copy, time
from torch.amp import GradScaler

EPOCHS         = 10
INITIAL_LR     = 3e-4
UNFREEZE_EPOCH = 4      # Warm-up 3 epochs → then fine-tune whole model
MIN_REAL_ACC   = 0.50   # Don't save unless Real class ≥ 50% (avoids degenerate models)

scaler    = GradScaler('cuda')
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=INITIAL_LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ── Auto-resume ─────────────────────────────────
best_acc     = 0.0
start_epoch  = 1
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_weights = copy.deepcopy(model.state_dict())

if CKPT_PATH.exists():
    print('🔄 Loading checkpoint for resume...')
    ckpt         = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    best_acc     = ckpt.get('val_accuracy', 0.0)
    start_epoch  = ckpt.get('epoch', 0) + 1
    history      = ckpt.get('history', history)
    best_weights = copy.deepcopy(model.state_dict())
    print(f'✅ Resumed from epoch {start_epoch - 1} | Best val acc: {best_acc*100:.2f}%')

if start_epoch > EPOCHS:
    print('✅ Training already complete!')
else:
    print(f'\n{"+"*55}')
    print(f'  Epochs: {start_epoch}–{EPOCHS} | BS={BATCH_SIZE} | LR={INITIAL_LR}')
    print(f'  Unfreeze backbone at epoch {UNFREEZE_EPOCH}')
    print(f'{"+"*55}')

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f'\n{"="*55}')
        print(f'  Epoch {epoch}/{EPOCHS}')
        print(f'{"="*55}')

        if epoch == UNFREEZE_EPOCH:
            print('🔓 Unfreezing backbone (differential LR)...')
            model.unfreeze_backbone()
            optimizer = optim.AdamW([
                {'params': model.backbone.features.parameters(),    'lr': INITIAL_LR * 0.1},
                {'params': model.backbone.classifier.parameters(), 'lr': INITIAL_LR},
            ], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS - UNFREEZE_EPOCH + 1, eta_min=1e-6
            )
            total_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f'  Trainable params now: {total_t:,}')

        t0 = time.time()
        tr_loss, tr_acc, tr_cls = train_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc, va_cls, _, _ = validate_epoch(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        dur    = time.time() - t0
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'\n  Train → Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}%')
        print(f'  Val   → Loss: {va_loss:.4f}  Acc: {va_acc*100:.2f}%')
        print(f'  Time : {dur:.0f}s  |  LR: {cur_lr:.2e}')
        print(f'  Per-class val accuracy:')
        for i, name in enumerate(CLASS_NAMES):
            print(f'    {name:>14}: {va_cls[i]*100:.2f}%')

        # Determine index of 'real' class (robust to label ordering)
        real_idx  = CLASS_NAMES.index('real') if 'real' in CLASS_NAMES else 0
        real_ok   = va_cls[real_idx] >= MIN_REAL_ACC

        if va_acc > best_acc and real_ok:
            best_acc     = va_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({
                'model_state_dict':    model.state_dict(),
                'num_classes':         NUM_CLASSES,
                'class_names':         CLASS_NAMES,
                'epoch':               epoch,
                'val_accuracy':        va_acc,
                'per_class_accuracy':  va_cls,
                'history':             history,
            }, CKPT_PATH)
            print(f'  💾 Saved best_model.pth  (Val Acc: {va_acc*100:.2f}%)')
        elif va_acc > best_acc:
            print(f'  ⚠️  Acc improved but Real acc {va_cls[real_idx]*100:.1f}% < {MIN_REAL_ACC*100:.0f}% — not saving')
        else:
            print(f'  📝 Best so far: {best_acc*100:.2f}%')

        torch.cuda.empty_cache()

    print(f'\n{"="*55}')
    print(f'  ✅ TRAINING COMPLETE!')
    print(f'  Best Val Accuracy : {best_acc*100:.2f}%')
    print(f'  Saved to          : {CKPT_PATH}')
    print(f'{"="*55}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 9 — Evaluation & Confusion Matrix
# ═══════════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(best_weights)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Evaluating'):
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
print(f'\n{"="*55}')
print(f'  FINAL EVALUATION')
print(f'{"="*55}')
print(f'  Overall Accuracy: {acc*100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Deepfake Detector', fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
cm_path = MODEL_SAVE / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'✅ Saved: {cm_path}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 10 — Training Curves
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history['train_acc']) + 1)

axes[0].plot(ep, [a*100 for a in history['train_acc']], 'b-o', label='Train', ms=4)
axes[0].plot(ep, [a*100 for a in history['val_acc']],   'r-o', label='Val',   ms=4)
axes[0].axhline(best_acc*100, color='g', ls='--', alpha=0.7, label=f'Best {best_acc*100:.1f}%')
axes[0].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_loss'], 'b-o', label='Train', ms=4)
axes[1].plot(ep, history['val_loss'],   'r-o', label='Val',   ms=4)
axes[1].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Deepfake Detector — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = MODEL_SAVE / 'training_curves.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'✅ Saved: {plot_path}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 11 — Export Model for Web App
# ═══════════════════════════════════════════════
from google.colab import files

EXPORT_PATH = MODEL_SAVE / 'deepfake_detector_export.pth'
torch.save({
    'model_state_dict': best_weights,
    'class_names':      CLASS_NAMES,
    'num_classes':      NUM_CLASSES,
    'img_size':         IMG_SIZE,
    'mean':             MEAN,
    'std':              STD,
    'val_accuracy':     best_acc,
    'model_arch':       'efficientnet_b0',
}, EXPORT_PATH)

size_mb = os.path.getsize(EXPORT_PATH) / 1e6
print(f'{"="*55}')
print(f'  MODEL EXPORT READY')
print(f'{"="*55}')
print(f'  File : deepfake_detector_export.pth')
print(f'  Size : {size_mb:.0f} MB')
print(f'  Acc  : {best_acc*100:.2f}%')
print(f'  Arch : EfficientNet-B0')
print(f'{"="*55}')
print()
print('📥 Downloading to your browser...')
files.download(str(EXPORT_PATH))
print('✅ Done!')

## 🌐 Web App Inference Snippet

Paste this into your Flask / FastAPI app:

```python
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ── Load checkpoint ──────────────────────────────
ckpt       = torch.load('models/deepfake_detector_export.pth', map_location='cpu')
CLASS_NAMES = ckpt['class_names']
IMG_SIZE    = ckpt['img_size']
MEAN        = ckpt['mean']
STD         = ckpt['std']

# ── Rebuild model ────────────────────────────────
model = models.efficientnet_b0(weights=None)
in_f  = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_f, 256),
    nn.ReLU(inplace=True),
    nn.BatchNorm1d(256),
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(256, len(CLASS_NAMES)),
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ── Transform ────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Predict ──────────────────────────────────────
def predict(img_path: str) -> dict:
    img  = Image.open(img_path).convert('RGB')
    x    = transform(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    pred = probs.argmax().item()
    return {
        'label':      CLASS_NAMES[pred],
        'confidence': round(probs[pred].item() * 100, 2),
        'scores':     {c: round(probs[i].item() * 100, 2)
                       for i, c in enumerate(CLASS_NAMES)}
    }
```